# Day 3 — Chunking, Metadata & Embedding Strategy

**Goal:** take yesterday's parsed chunks and optimize them for retrieval.
We'll compare 3 chunking strategies, add LLM-extracted metadata, and shop for
an embedding model with an eval set.

By the end of today, you can justify every design decision in your pipeline
with numbers.

## 1. Setup — reload the cursed-pack parsing from Day 2

Day 3 is about retrieval quality engineering: once parsing is stable, chunk design and metadata become the main levers.

**Big idea**
- Chunking is not just splitting text. It defines the unit of retrieval, which directly shapes recall, precision, and prompt quality.


In [ ]:
# Reload the shared utilities and models we need for chunking experiments.
# This notebook is about controlled retrieval experiments, so reproducible setup matters.
from dotenv import load_dotenv
load_dotenv()

import json
import re
from pathlib import Path
from dataclasses import asdict

from utils import (
    Chunk, make_chunk_id, add_chunks, recall_at_k,
    get_chroma_client, get_llm_client
)
from sentence_transformers import SentenceTransformer

chroma = get_chroma_client("./chroma_db")
client, generate = get_llm_client()
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


# We'll use the PEP corpus (clean, easier to isolate chunking effects)
# and the SOP from the cursed pack (hierarchy matters).

def read_peps() -> list[dict]:
    docs = []
    for p in sorted(Path("./datasets/day1").glob("*.md")):
        docs.append({"source": p.name, "text": p.read_text()})
    return docs


peps = read_peps()
print(f"Loaded {len(peps)} PEPs totalling {sum(len(d['text']) for d in peps):,} chars")


## 2. Three chunking strategies

We'll build each as a pure function `chunk(text, source) -> list[Chunk]` so
they're comparable.

### Strategy A — Fixed-size (Day 1 baseline)
Just 2000-char windows with 200-char overlap. Cuts sentences mid-word.

We compare multiple chunking strategies because no single method wins on every corpus.

**What to watch for**
- Fixed windows are simple and consistent.
- Recursive chunking respects natural boundaries better.
- Semantic chunking tries to keep ideas together even when formatting is weak.


In [ ]:
# Fixed-size chunking is the baseline: simple, fast, and easy to reason about.
# We keep strategy metadata on each chunk so later evaluation can trace how it was produced.
def chunk_fixed(text: str, source: str, size: int = 2000, overlap: int = 200) -> list[Chunk]:
    out = []
    i = 0
    idx = 0
    while i < len(text):
        piece = text[i : i + size]
        out.append(Chunk(
            id=make_chunk_id(source, idx, piece),
            text=piece,
            source=source,
            content_type="prose",
            extra={"strategy": "fixed"},
        ))
        i += size - overlap
        idx += 1
    return out


### Strategy B — Recursive split on boundaries

Try to split on paragraph breaks first, then sentences, then words. This is
what LangChain's `RecursiveCharacterTextSplitter` does under the hood.

Recursive splitting is a practical middle ground between naive slicing and expensive semantic methods.

**Reasoning note**
- The algorithm prefers coarse boundaries first, then falls back to finer ones only when needed.
- That keeps chunks readable without letting them grow too large.


In [ ]:
# Recursive splitting prefers natural boundaries before falling back to raw size limits.
# That usually yields chunks that are easier for both retrieval and humans to interpret.
def recursive_split(text: str, separators: list[str], max_len: int) -> list[str]:
    """Split by the first separator that produces small-enough pieces.

    If no separator works, fall back to character-level slicing.
    """
    if len(text) <= max_len:
        return [text]

    for sep in separators:
        if sep == "":
            # Last resort — chop by size
            return [text[i : i + max_len] for i in range(0, len(text), max_len)]
        if sep in text:
            parts = text.split(sep)
            # Reassemble: glue parts back together until a piece would exceed max_len
            out = []
            buf = ""
            for part in parts:
                candidate = (buf + sep + part) if buf else part
                if len(candidate) <= max_len:
                    buf = candidate
                else:
                    if buf:
                        out.append(buf)
                    # If a single part is still too big, recurse on it
                    if len(part) > max_len:
                        out.extend(recursive_split(part, separators[1:], max_len))
                        buf = ""
                    else:
                        buf = part
            if buf:
                out.append(buf)
            return out
    return [text]


def chunk_recursive(text: str, source: str, max_len: int = 1500) -> list[Chunk]:
    pieces = recursive_split(text, ["\n\n", "\n", ". ", " ", ""], max_len)
    return [
        Chunk(
            id=make_chunk_id(source, i, p),
            text=p,
            source=source,
            content_type="prose",
            extra={"strategy": "recursive"},
        )
        for i, p in enumerate(pieces)
    ]


### Strategy C — Semantic chunking

Embed sentences; split where similarity between adjacent sentences drops
below a threshold. This keeps semantically related sentences together.

It's slower (one embedding per sentence) but produces more coherent chunks.

Semantic chunking tries to group adjacent sentences that belong together conceptually.

**Tradeoff**
- It can produce more meaningful chunks, but it is more computationally involved and can be less predictable on small corpora.


In [ ]:
# Semantic chunking uses sentence-level similarity to decide where topics naturally break.
# It is more adaptive than fixed windows, but also more complex and computationally heavier.
import numpy as np


def chunk_semantic(text: str, source: str, drop_percentile: float = 90) -> list[Chunk]:
    # Split into sentences (naively — '. ' is good enough for English)
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if len(sentences) < 3:
        return [Chunk(
            id=make_chunk_id(source, 0, text),
            text=text, source=source,
            content_type="prose",
            extra={"strategy": "semantic"},
        )]

    embs = st_model.encode(sentences, convert_to_numpy=True)

    # Cosine similarity between consecutive sentences
    def cos(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    sims = [cos(embs[i], embs[i + 1]) for i in range(len(embs) - 1)]
    # A *drop* in similarity = a good place to split. Use percentile on distances (1 - sim).
    distances = [1 - s for s in sims]
    threshold = float(np.percentile(distances, drop_percentile))

    chunks = []
    buf = [sentences[0]]
    idx = 0
    for i, dist in enumerate(distances):
        if dist > threshold:
            # End current chunk
            text_piece = " ".join(buf)
            chunks.append(Chunk(
                id=make_chunk_id(source, idx, text_piece),
                text=text_piece, source=source,
                content_type="prose",
                extra={"strategy": "semantic"},
            ))
            idx += 1
            buf = [sentences[i + 1]]
        else:
            buf.append(sentences[i + 1])
    if buf:
        text_piece = " ".join(buf)
        chunks.append(Chunk(
            id=make_chunk_id(source, idx, text_piece),
            text=text_piece, source=source,
            content_type="prose",
            extra={"strategy": "semantic"},
        ))
    return chunks


## 3. Build an eval set

10 hand-written Q&A pairs against the PEP corpus. For each, we record which
source file must be retrieved. This is a *retrieval* eval — we're measuring
whether the right document comes back, not whether the answer is generated
correctly. (That's Day 4.)

An eval set turns opinions about chunking into measurable results.

**Important habit**
- Retrieval changes should be judged against representative questions, not just one or two anecdotal prompts.


In [ ]:
# A small eval set is enough to compare strategies as long as the questions are representative.
# Explicit expected sources make recall measurable instead of subjective.
eval_set = [
    {"q": "What indentation does PEP 8 recommend?",                              "source": "pep-0008.md"},
    {"q": "What is the maximum line length per PEP 8?",                          "source": "pep-0008.md"},
    {"q": "What does the Zen of Python say about explicit vs implicit?",         "source": "pep-0020.md"},
    {"q": "How should one-line docstrings be formatted?",                        "source": "pep-0257.md"},
    {"q": "What are the rules for multi-line docstrings?",                       "source": "pep-0257.md"},
    {"q": "What is an absolute import?",                                          "source": "pep-0328.md"},
    {"q": "Why were relative imports deprecated at one point?",                  "source": "pep-0328.md"},
    {"q": "What are type hints?",                                                 "source": "pep-0484.md"},
    {"q": "How does PEP 484 handle generic types?",                              "source": "pep-0484.md"},
    {"q": "How should imports be grouped?",                                       "source": "pep-0008.md"},
]
print(f"Eval set: {len(eval_set)} questions")


## 4. Bake-off: three strategies, one eval

The bake-off keeps the embedding model fixed so we isolate the effect of chunking.

**Experimental design**
- Change one major variable at a time.
- Otherwise it becomes hard to tell whether the improvement came from chunking, embeddings, or luck.


In [ ]:
# Build a fresh collection per strategy so comparisons stay isolated and repeatable.
# The evaluation function tests retrieval, not generation, because that is the variable under study.
def build_collection(name: str, chunks: list[Chunk]) -> "Collection":
    try:
        chroma.delete_collection(name)
    except Exception:
        pass
    coll = chroma.create_collection(name=name)
    embs = st_model.encode([c.text for c in chunks], show_progress_bar=False)
    add_chunks(coll, chunks, embeddings=embs.tolist())
    return coll


def eval_strategy(name: str, chunker) -> dict:
    # Build chunks from all PEPs
    chunks = []
    for doc in peps:
        chunks.extend(chunker(doc["text"], doc["source"]))

    coll = build_collection(f"day3_{name}", chunks)

    hits_correct = 0
    for item in eval_set:
        q_vec = st_model.encode([item["q"]])[0].tolist()
        res = coll.query(query_embeddings=[q_vec], n_results=5)
        sources = [md["source"] for md in res["metadatas"][0]]
        if item["source"] in sources:
            hits_correct += 1

    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "recall_at_5": hits_correct / len(eval_set),
    }


results = []
for name, fn in [
    ("fixed",     lambda t, s: chunk_fixed(t, s)),
    ("recursive", lambda t, s: chunk_recursive(t, s)),
    ("semantic",  lambda t, s: chunk_semantic(t, s)),
]:
    r = eval_strategy(name, fn)
    results.append(r)
    print(f"{r['strategy']:10s} → {r['n_chunks']:4d} chunks,  recall@5 = {r['recall_at_5']:.2f}")


## 5. Metadata extraction with an LLM

Rich metadata unlocks filtered retrieval. Rather than hand-labelling, ask
Claude to extract structured fields from each chunk.

We'll extract:
- `document_type` (one of: standard_doc, procedure, announcement)
- `topic_tags` (up to 3 short tags)
- `has_code` (bool)
- `year_mentioned` (if any specific year appears in the text)

Metadata makes retrieval more controllable. It lets you filter or bias search using structured signals beyond pure vector similarity.

**Examples**
- Document type
- Topic tags
- Whether the chunk includes code
- Time-related fields such as a year


In [ ]:
# The metadata prompt asks for a compact schema that is actually useful for filtering later.
# Narrow schemas are easier to validate and more robust than open-ended extraction.
METADATA_PROMPT = """Extract structured metadata from this text chunk.

Return a JSON object with keys:
- "document_type": one of ["standard_doc", "procedure", "tutorial", "announcement", "other"]
- "topic_tags": array of up to 3 short lowercase tags (e.g., "indentation", "imports")
- "has_code": boolean (true if the chunk contains code examples)
- "year_mentioned": integer or null (e.g., 2006 if a year is mentioned)

Return ONLY the JSON, no other text.

Text:
{text}
"""


def extract_metadata(text: str) -> dict:
    raw = generate(
        system="You extract structured metadata. Return valid JSON only.",
        user=METADATA_PROMPT.format(text=text[:2000]),
        max_tokens=200,
    )
    # Strip code fences if Claude adds them
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"document_type": "other", "topic_tags": [], "has_code": False, "year_mentioned": None}


# Demo on 3 chunks
demo_chunks = chunk_recursive(peps[0]["text"], peps[0]["source"])[:3]
for c in demo_chunks:
    md = extract_metadata(c.text)
    print(f"Chunk preview: {c.text[:80]}...")
    print(f"  metadata: {md}")
    print()


### Apply metadata to a full corpus

For a real engagement, run this once and cache the results. It costs money
(~$0.001 per chunk with Claude Haiku) but pays off every query.

Applying metadata across a full corpus is where extraction quality starts to matter.

**What to look for**
- Missing or noisy metadata can silently reduce the usefulness of filters.
- Storing metadata in consistent formats makes later querying much simpler.


In [ ]:
# Apply the same metadata extraction logic across the winning chunk set.
# We store normalized values so Chroma filters remain simple and predictable.
# Keep the winning chunker (recursive for PEPs based on recall@5)
winner_chunks = []
for doc in peps:
    winner_chunks.extend(chunk_recursive(doc["text"], doc["source"]))

print(f"Extracting metadata for {len(winner_chunks)} chunks...")
for c in winner_chunks:
    md = extract_metadata(c.text)
    c.document_type = md.get("document_type")
    if md.get("topic_tags"):
        c.extra["topic_tags"] = ",".join(md["topic_tags"])
    if md.get("has_code") is not None:
        c.extra["has_code"] = bool(md["has_code"])
    year = md.get("year_mentioned")
    if year:
        c.extra["year_mentioned"] = int(year[0] if isinstance(year, list) else year)

print("Metadata attached. Sample:")
for c in winner_chunks[:3]:
    print(f"  {c.source} [{c.document_type}] tags={c.extra.get('topic_tags')} "
          f"has_code={c.extra.get('has_code')}")


## 6. Filtered retrieval using metadata

Now the payoff. Queries can restrict to a subset of the corpus.

Filtered retrieval is useful when users ask constrained questions like 'show only code examples' or 'only procedures from 2024'.

**Design insight**
- Metadata filters can increase precision, but they can also hide relevant chunks if extraction is wrong.


In [ ]:
# Index the metadata-enriched chunks into a separate collection so we can test filtered retrieval directly.
# Keeping collections separate avoids accidental cross-contamination during experiments.
try:
    chroma.delete_collection("day3_metadata")
except Exception:
    pass

coll_md = chroma.create_collection(name="day3_metadata")
embs = st_model.encode([c.text for c in winner_chunks], show_progress_bar=False)
add_chunks(coll_md, winner_chunks, embeddings=embs.tolist())
print(f"Indexed {coll_md.count()} chunks with metadata.")


In [ ]:
# Retrieval with optional metadata filters lets us test whether structured fields improve precision.
# The output is intentionally compact so it is easy to scan during interactive experiments.
def retrieve_filtered(query: str, where: dict | None = None, k: int = 3) -> list[dict]:
    q_vec = st_model.encode([query])[0].tolist()
    kwargs = {"query_embeddings": [q_vec], "n_results": k}
    if where:
        kwargs["where"] = where
    res = coll_md.query(**kwargs)
    out = []
    for i in range(len(res["ids"][0])):
        out.append({
            "text": res["documents"][0][i][:120] + "...",
            "source": res["metadatas"][0][i].get("source"),
            "has_code": res["metadatas"][0][i].get("has_code"),
            "tags": res["metadatas"][0][i].get("topic_tags"),
        })
    return out


# Query 1: Normal retrieval
print("Query: 'how to format imports', no filter")
for h in retrieve_filtered("how to format imports"):
    print(f"  [{h['source']}] tags={h['tags']}  {h['text']}")
print()

# Query 2: Only chunks with code examples
print("Query: 'how to format imports', filter has_code=True")
for h in retrieve_filtered("how to format imports", where={"has_code": True}):
    print(f"  [{h['source']}] tags={h['tags']}  {h['text']}")
print()

# Query 3: Only chunks from pep-0008
print("Query: 'indentation', filter source=pep-0008.md")
for h in retrieve_filtered("indentation", where={"source": "pep-0008.md"}):
    print(f"  [{h['source']}] tags={h['tags']}  {h['text']}")


## 7. Embedding bake-off

Compare two free models against each other on our eval set. If you have
budget, add OpenAI `text-embedding-3-small` — often 5–10 points better on
complex queries.

**Models:**
- `sentence-transformers/all-MiniLM-L6-v2` (384-dim, 80MB, fastest)
- `BAAI/bge-large-en-v1.5` (1024-dim, 1.3GB, much stronger)

Embedding bake-offs help separate model choice from pipeline design.

**How to interpret results**
- A stronger model may help, but gains are often smaller than expected if chunking and metadata are already doing most of the work.


In [ ]:
# This embedding bake-off holds chunking fixed while swapping only the encoder.
# That separation helps us tell whether model upgrades are worth their extra cost or size.
def eval_embedding(model_name: str, chunker_fn) -> dict:
    model = SentenceTransformer(model_name)
    chunks = []
    for doc in peps:
        chunks.extend(chunker_fn(doc["text"], doc["source"]))

    coll_name = f"day3_emb_{re.sub(r'[^a-z0-9]', '_', model_name.lower())}"
    try:
        chroma.delete_collection(coll_name)
    except Exception:
        pass

    coll = chroma.create_collection(name=coll_name)
    embs = model.encode([c.text for c in chunks], show_progress_bar=False)
    add_chunks(coll, chunks, embeddings=embs.tolist())

    hits = 0
    for item in eval_set:
        q_vec = model.encode([item["q"]])[0].tolist()
        res = coll.query(query_embeddings=[q_vec], n_results=5)
        if item["source"] in [md["source"] for md in res["metadatas"][0]]:
            hits += 1

    return {"model": model_name, "recall_at_5": hits / len(eval_set)}


print("MiniLM (baseline):")
r = eval_embedding("sentence-transformers/all-MiniLM-L6-v2", chunk_recursive)
print(f"  recall@5 = {r['recall_at_5']:.2f}")

# BGE-large is a 1.3GB download — uncomment when you're ready to wait
# print("\nBGE-large:")
# r = eval_embedding("BAAI/bge-large-en-v1.5", chunk_recursive)
# print(f"  recall@5 = {r['recall_at_5']:.2f}")


## 8. Reflection

**Questions to answer in your team stand-up tomorrow:**

1. Which chunking strategy won on your eval set, and by how much?
2. What metadata fields would be most useful for *your* client's domain?
3. If BGE beats MiniLM by 3 points but takes 10x longer to embed, do you
   use it? What's the decision rule?
4. Bonus: rerun the eval with `k=1` and `k=10`. Where does the strategy
   choice matter most?

Tomorrow we add scraping, hybrid search, reranking, and generation eval.

Reflection should connect architecture choices to evaluation evidence.

**Ask yourself**
- Which chunking strategy produced the most interpretable failures?
- Which metadata fields were actually useful versus merely available?
